# Materialize Gold/Staging_Gold vw_* as Delta tables

Spark views are **not** queryable via the SQL Analytics Endpoint.
This notebook drops those Spark views and recreates the same names as **managed Delta tables**
so backend APIs / T-SQL can bind them.

After incremental Silver/Gold runs, re-run this notebook to refresh `vw_*` rollups from the **full current** `rpt_unified_ad_performance` (derived overwrite; does not delete Gold history).


In [ ]:
from pyspark.sql import functions as F
from notebookutils import mssparkutils

OBJECTS = [
    "Gold.vw_campaign_performance",
    "Gold.vw_adset_performance",
    "Gold.vw_ad_performance",
    "Gold.vw_unified_ad_performance",
    "Staging_Gold.vw_campaign_performance",
    "Staging_Gold.vw_adset_performance",
    "Staging_Gold.vw_ad_performance",
    "Staging_Gold.vw_unified_ad_performance",
]

spark.sql("CREATE SCHEMA IF NOT EXISTS Gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS Staging_Gold")

for obj in OBJECTS:
    spark.sql(f"DROP VIEW IF EXISTS {obj}")
    spark.sql(f"DROP TABLE IF EXISTS {obj}")
    print("cleared", obj)


In [ ]:
SRC = "Gold.rpt_unified_ad_performance"

campaign_sql = f"""
SELECT
  platform, full_date, year, month, month_name, day_name,
  MAX(tenant_id) AS tenant_id,
  MAX(connector_id) AS connector_id,
  account_id,
  CAST(NULL AS STRING) AS customer_id,
  MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  MAX(campaign_status) AS campaign_status,
  MAX(campaign_channel_or_objective) AS campaign_channel_or_objective,
  MAX(daily_budget_inr) AS daily_budget_inr,
  SUM(impressions) AS impressions, SUM(reach) AS reach,
  SUM(clicks) AS clicks, SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend) / SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend) / SUM(impressions)) * 1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend) / SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT adset_id) AS adset_count,
  COUNT(DISTINCT ad_id) AS ad_count
FROM {SRC}
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id
"""

adset_sql = f"""
SELECT
  platform, full_date, year, month, month_name, day_name,
  MAX(tenant_id) AS tenant_id,
  MAX(connector_id) AS connector_id,
  account_id,
  CAST(NULL AS STRING) AS customer_id,
  MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  adset_id, MAX(adset_name) AS adset_name, MAX(adset_status) AS adset_status,
  MAX(optimization_goal) AS optimization_goal,
  MAX(age_range) AS age_range, MAX(geo_cities) AS geo_cities, MAX(geo_regions) AS geo_regions,
  SUM(impressions) AS impressions, SUM(reach) AS reach,
  SUM(clicks) AS clicks, SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend) / SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend) / SUM(impressions)) * 1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend) / SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT ad_id) AS ad_count
FROM {SRC}
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id, adset_id
"""

def write_delta(name, df):
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(name)
    )
    print("[OK]", name, "type=MANAGED-delta rows", spark.table(name).count())

write_delta("Gold.vw_campaign_performance", spark.sql(campaign_sql))
write_delta("Gold.vw_adset_performance", spark.sql(adset_sql))
write_delta(
    "Gold.vw_ad_performance",
    spark.table(SRC).withColumn("customer_id", F.lit(None).cast("string")),
)
write_delta("Gold.vw_unified_ad_performance", spark.table(SRC))

write_delta("Staging_Gold.vw_campaign_performance", spark.sql(campaign_sql))
write_delta("Staging_Gold.vw_adset_performance", spark.sql(adset_sql))
write_delta(
    "Staging_Gold.vw_ad_performance",
    spark.table(SRC).withColumn("customer_id", F.lit(None).cast("string")),
)
write_delta("Staging_Gold.vw_unified_ad_performance", spark.table(SRC))


In [ ]:
wanted = ["connector_id", "account_id", "tenant_id", "customer_id"]
lines = []
for obj in [
    "Gold.vw_campaign_performance",
    "Gold.vw_adset_performance",
    "Gold.vw_ad_performance",
    "Staging_Gold.vw_campaign_performance",
    "Staging_Gold.vw_adset_performance",
    "Staging_Gold.vw_ad_performance",
]:
    meta = {
        r.col_name: r.data_type
        for r in spark.sql(f"DESCRIBE EXTENDED {obj}").collect()
        if r.col_name in ("Type", "Provider")
    }
    cols = set(spark.table(obj).columns)
    lines.append(
        f"{obj} Type={meta.get('Type')} Provider={meta.get('Provider')} "
        f"ids={[c for c in wanted if c in cols]} rows={spark.table(obj).count()}"
    )

text = "\n".join(lines)
print(text)
mssparkutils.fs.put("Files/Development/Gold/exports/materialize_vw_delta_check.txt", text, True)
print("MATERIALIZE_VW_DELTA_COMPLETE")
